# ЛР 03.1 — Train/Validation и переобучение (TODO)

## Цель
- взять все candidate feature set из ЛР 01 как гипотезы;
- сравнить `full` и неполные feature set на `train` и `validation`;
- увидеть `generalization gap` на понятных цифрах;
- выбрать feature set отдельно для каждой модели;
- сохранить этот выбор как явный артефакт для второго ноутбука;
- посмотреть, как один гиперпараметр меняет поведение модели.

## Что важно
- высокая метрика на `train` еще не означает хорошую работу на новых данных;
- `test` пока не используется для выбора модели;
- в этой ЛР candidate feature set из ЛР 01 переоцениваются заново на текущем split;
- базовый маршрут честен по отношению к `test`, но упрощает workflow за счет повторного использования `validation`;
- на продвинутом треке этот маршрут можно усилить через nested CV или отдельный selection split.


In [ ]:
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '03-overfitting-validation-and-hyperparameter-tuning',
    cwd.parent / '03-overfitting-validation-and-hyperparameter-tuning',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError(
        'Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 03 или из корня репозитория.'
    )

spec = importlib.util.spec_from_file_location('lab03_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', context='talk')


## Шаг 1. Загрузка данных и candidate feature set из ЛР 01

На этом шаге:
- загружаем оба датасета курса;
- берем все candidate feature set из `feature_sets_wrapper_embedded.json`;
- один раз делим данные на `train`, `validation`, `test` в схеме `60/20/20`.

`test` пока откладываем в сторону: он понадобится только в конце второго ноутбука.


In [ ]:
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()

prepared = {}
selection_rows = []

for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_valid, x_test, y_train, y_valid, y_test = lab.train_valid_test_split_stratified(x, y)
    feature_set_names = lab.list_feature_set_names(feature_sets, dataset_name)

    prepared[dataset_name] = {
        'x_train': x_train,
        'x_valid': x_valid,
        'x_test': x_test,
        'y_train': y_train,
        'y_valid': y_valid,
        'y_test': y_test,
        'feature_set_names': feature_set_names,
    }

    selection_rows.append(
        {
            'dataset': dataset_name,
            'candidate_feature_sets': ', '.join(feature_set_names),
            'n_train': len(x_train),
            'n_validation': len(x_valid),
            'n_test': len(x_test),
        }
    )

candidate_feature_sets = pd.DataFrame(selection_rows).sort_values('dataset').reset_index(drop=True)
candidate_feature_sets


## Шаг 2. Baseline-аудит по всем candidate feature set

Здесь мы для каждого dataset сравниваем:
- `full`;
- все неполные candidate feature set;
- обе модели `LogisticRegression` и `RandomForest`.

На этом этапе мы честно смотрим только на `train` и `validation`.


In [ ]:
audit_rows = []

for dataset_name, ctx in prepared.items():
    for feature_set_name in ctx['feature_set_names']:
        selected_features = lab.get_feature_set_features(feature_sets, dataset_name, feature_set_name)
        selector = lab.PreprocessedFeatureSelector(selected_features=selected_features).fit(
            ctx['x_train'], ctx['y_train']
        )
        x_train_sel = selector.transform(ctx['x_train'])
        x_valid_sel = selector.transform(ctx['x_valid'])

        for model_name, base_model in lab.make_default_models().items():
            _, fit_time_sec, train_metrics, valid_metrics = lab.measure_fit_and_split_metrics(
                clone(base_model),
                x_train_sel,
                ctx['y_train'],
                x_valid_sel,
                ctx['y_valid'],
            )

            for split_name, metrics in [
                ('train', train_metrics),
                ('validation', valid_metrics),
            ]:
                audit_rows.append(
                    {
                        'dataset': dataset_name,
                        'feature_set': feature_set_name,
                        'model': model_name,
                        'split': split_name,
                        'accuracy': metrics['accuracy'],
                        'f1': metrics['f1'],
                        'roc_auc': metrics['roc_auc'],
                        'fit_time_sec': fit_time_sec,
                    }
                )

generalization_audit = pd.DataFrame(audit_rows)
generalization_audit['split'] = pd.Categorical(
    generalization_audit['split'],
    categories=['train', 'validation'],
    ordered=True,
)
generalization_audit = generalization_audit.sort_values(
    ['dataset', 'feature_set', 'model', 'split']
).reset_index(drop=True)
generalization_audit


## Шаг 3. Выбор feature set отдельно для каждой модели

Правило выбора для каждой пары `dataset + model`:
- максимум `validation_f1`;
- затем минимум `abs(train_f1 - validation_f1)`;
- затем предпочтение неполному набору;
- затем лексикографически меньший `feature_set`.

Небольшой отрицательный gap допустим. Если `validation` чуть выше `train`, это не ошибка само по себе.
Результат этого шага мы сохраняем как `model_feature_set_decisions.csv`, чтобы второй ноутбук читал явный входной контракт, а не пересчитывал выбор скрыто.


In [ ]:
gap_summary = lab.build_generalization_selection_summary(generalization_audit).reset_index(drop=True)
model_feature_set_decisions = lab.build_model_feature_set_decisions(generalization_audit)

display(gap_summary)
model_feature_set_decisions


## Шаг 4. Простые validation curves

Мы не строим полный поиск параметров в этом ноутбуке.
Вместо этого смотрим на один параметр за раз:
- для `LogisticRegression` меняем `C`;
- для `RandomForest` меняем `max_depth`.

Validation curves строятся на feature set, который был выбран для конкретной модели.


In [ ]:
curve_rows = []

for dataset_name, ctx in prepared.items():
    for model_name, base_model in lab.make_default_models().items():
        feature_set_name = model_feature_set_decisions.loc[
            (model_feature_set_decisions['dataset'] == dataset_name)
            & (model_feature_set_decisions['model'] == model_name),
            'selected_feature_set',
        ].iloc[0]
        selected_features = lab.get_feature_set_features(feature_sets, dataset_name, feature_set_name)
        selector = lab.PreprocessedFeatureSelector(selected_features=selected_features).fit(
            ctx['x_train'], ctx['y_train']
        )
        x_train_sel = selector.transform(ctx['x_train'])
        x_valid_sel = selector.transform(ctx['x_valid'])

        hyperparameter, param_grid = lab.VALIDATION_CURVE_GRIDS[model_name]
        for param_value in param_grid:
            model = clone(base_model)
            model.set_params(**{hyperparameter: param_value})
            _, _, train_metrics, valid_metrics = lab.measure_fit_and_split_metrics(
                model,
                x_train_sel,
                ctx['y_train'],
                x_valid_sel,
                ctx['y_valid'],
            )

            for split_name, metrics in [
                ('train', train_metrics),
                ('validation', valid_metrics),
            ]:
                curve_rows.append(
                    {
                        'dataset': dataset_name,
                        'feature_set': feature_set_name,
                        'model': model_name,
                        'hyperparameter': hyperparameter,
                        'param_value': lab.format_param_value(param_value),
                        'split': split_name,
                        'accuracy': metrics['accuracy'],
                        'f1': metrics['f1'],
                        'roc_auc': metrics['roc_auc'],
                    }
                )

validation_curve_results = pd.DataFrame(curve_rows)
validation_curve_results['split'] = pd.Categorical(
    validation_curve_results['split'],
    categories=['train', 'validation'],
    ordered=True,
)
validation_curve_results = validation_curve_results.sort_values(
    ['dataset', 'model', 'split', 'param_value']
).reset_index(drop=True)
validation_curve_results


In [ ]:
plot_df = validation_curve_results.copy()

grid = sns.relplot(
    data=plot_df,
    x='param_value',
    y='f1',
    hue='split',
    col='model',
    row='dataset',
    kind='line',
    marker='o',
    facet_kws={'sharey': False, 'sharex': False},
)
grid.set_axis_labels('Значение гиперпараметра', 'F1')
grid.set_titles(row_template='{row_name}', col_template='{col_name}')
plt.show()


## Самостоятельное изучение по ходу работы

Заполните своими словами:
- где вы увидели самый явный пример переобучения;
- где train-метрика выглядела слишком оптимистично;
- где validation оказалась не хуже train и почему это не является ошибкой;
- почему feature set в этой ЛР выбирается отдельно для каждой модели;
- какое didactic shortcut использован в базовом маршруте и как его можно усилить на продвинутом треке.

При необходимости вынесите разбор в:
- `study-notes/overfitting-vs-underfitting.md`
- `study-notes/train-validation-test-split.md`
- `study-notes/validation-reuse-vs-nested-cv.md`


## Контрольные точки

Перед переходом ко второму ноутбуку проверьте:
1. Есть таблица `generalization_audit` с раздельными метриками для `train` и `validation`.
2. Есть таблица `model_feature_set_decisions` с выбором feature set отдельно для каждой модели.
3. Есть таблица `validation_curve_results`.
4. `test` еще не использовался для выбора модели или параметров.


In [ ]:
required_generalization_columns = {
    'dataset',
    'feature_set',
    'model',
    'split',
    'accuracy',
    'f1',
    'roc_auc',
    'fit_time_sec',
}
required_decision_columns = set(lab.MODEL_FEATURE_SET_DECISION_COLUMNS)
required_curve_columns = {
    'dataset',
    'feature_set',
    'model',
    'hyperparameter',
    'param_value',
    'split',
    'accuracy',
    'f1',
    'roc_auc',
}

# TODO(обязательно):
# 1) Проверьте, что generalization_audit, model_feature_set_decisions и validation_curve_results содержат нужные колонки.
# 2) Сохраните все три DataFrame в CSV внутри outputs/.
# 3) Используйте model_feature_set_decisions в narrative-блоках и объясните tie-break rules.

raise NotImplementedError(
    'Самостоятельный блок не завершен: сохраните outputs/generalization_audit.csv, outputs/model_feature_set_decisions.csv и outputs/validation_curve_results.csv.'
)
